# 0. Imports and Configuration

In [ ]:
import re
import numpy as np
import pandas as pd
import joblib
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
from scipy.sparse import csr_matrix

In [ ]:
RANDOM_STATE = 42
TEST_SIZE = 0.20
N_COMPONENTS = 50
K = 10
RELEVANCE_THRESHOLD = 0.10

# Identical scoring weights for KNN and SVD
SIMILARITY_WEIGHT = 0.60
RATING_WEIGHT = 0.40

# Movie recommendation weights
GENRE_WEIGHT = 0.50
ACTOR_WEIGHT = 0.25
DIRECTOR_WEIGHT = 0.25

TOP_N_GENRES = 3
TOP_N_PEOPLE = 3
TOP_N_MOVIES = 3
GROUPS = {
    "genre": (GENRE_WEIGHT, "genre_set"),
    "actor": (ACTOR_WEIGHT, "actor_set"),
    "director": (DIRECTOR_WEIGHT, "director_set"),
}

MODEL_PATH = Path("movie_recommender.joblib")

required_columns = [
    "tconst",
    "primaryTitle",
    "genres",
    "actor",
    "director",
    "movie_category",
    "weightedRating",
]

In [ ]:
df = pd.read_csv("complete_movie_data.csv")
agg_dict = {
    'weightedRating': 'max',
    'numVotes': 'max',
    'genres': 'max',
    'movie_category': 'max',
    'primaryTitle': 'max',
    'director': 'max',
    'actor': '|'.join
}

final_df = df.groupby(['tconst', 'startYear'], as_index=False).agg(agg_dict)
missing_columns = [
    col for col in required_columns
    if col not in final_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

df = final_df[required_columns].copy()

# 1. Final Preprocessing

# 1.1 Helper Functions

In [ ]:
def parse_genres(value):
    """Parse space-separated genre names."""
    if pd.isna(value):
        return []

    return sorted({
        item.strip().lower()
        for item in str(value).split()
        if item.strip()
    })


def parse_entities(value):
    """
    Parse actor/director names separated by | or ;.

    Names are kept intact: for example, 'walton goggins'
    remains one person rather than two separate tokens.
    """
    if pd.isna(value):
        return []

    return sorted({
        item.strip().lower()
        for item in re.split(r"[|;]", str(value))
        if item.strip()
    })

# 1.2 Run Preproc

In [ ]:
df["movie_category"] = (
    df["movie_category"]
    .astype("string")
    .str.strip()
)
df = df.dropna(
    subset=["tconst", "primaryTitle", "movie_category"]
).copy()
df = df.drop_duplicates(subset="tconst").reset_index(drop=True)
df["weightedRating"] = pd.to_numeric(
    df["weightedRating"],
    errors="coerce"
)
df["genre_list"] = df["genres"].apply(parse_genres)
df["actor_list"] = df["actor"].apply(parse_entities)
df["director_list"] = df["director"].apply(parse_entities)
if df["weightedRating"].notna().any():
    df["rating_score"] = (
        df["weightedRating"]
        .rank(method="average", pct=True)
    )
else:
    df["rating_score"] = 0.5

df["rating_score"] = df["rating_score"].fillna(0.5)

df["tokens"] = df.apply(
    lambda row: (
        [f"category:{row['movie_category'].lower()}"]
        + [f"genre:{g}" for g in row["genre_list"]]
        + [f"actor:{a}" for a in row["actor_list"]]
        + [f"director:{d}" for d in row["director_list"]]
    ),
    axis=1,
)

# 2 Feature Matrix Creation

In [ ]:
encoder = MultiLabelBinarizer(sparse_output=True)

X = encoder.fit_transform(df["tokens"]).tocsr()

feature_index = {
    token: i
    for i, token in enumerate(encoder.classes_)
}

# KNN operates on the original sparse feature matrix.
X_knn = normalize(X, norm="l2", axis=1)

# SVD learns a lower-dimensional representation of the same data.
n_components = min(
    N_COMPONENTS,
    X.shape[0] - 1,
    X.shape[1] - 1
)

if n_components < 1:
    raise ValueError(
        "Not enough movies or features to fit SVD."
    )

svd = TruncatedSVD(
    n_components=n_components,
    random_state=RANDOM_STATE
)

X_svd = svd.fit_transform(X)

# Normalize movie vectors so cosine similarity is comparable.
X_svd = normalize(X_svd, norm="l2", axis=1)

# Cache movie indices for each genre, actor, and director.
genre_to_indices = {}
actor_to_indices = {}
director_to_indices = {}

for i, row in df.iterrows():
    for genre in row["genre_list"]:
        genre_to_indices.setdefault(genre, []).append(i)

    for actor in row["actor_list"]:
        actor_to_indices.setdefault(actor, []).append(i)

    for director in row["director_list"]:
        director_to_indices.setdefault(director, []).append(i)

# 3. Similarity functions and mapping

In [ ]:
def get_movie_vectors(method):
    """Return the movie vectors for the selected method."""
    method = method.upper()

    if method == "KNN":
        return X_knn

    if method == "SVD":
        return X_svd

    raise ValueError("method must be 'KNN' or 'SVD'")


def profile_vector(movie_indices, method):
    """
    Represent a group of movies by its mean feature vector.

    KNN: mean vector in the original feature space.
    SVD: mean vector in the latent space.
    """
    if len(movie_indices) == 0:
        return None

    vectors = get_movie_vectors(method)
    profile = np.asarray(
        vectors[movie_indices].mean(axis=0)
    ).reshape(1, -1)

    return normalize(profile, norm="l2", axis=1)


def similarity_to_profile(candidate_indices, target_indices, method):
    """
    Compute candidate-profile similarity to a target profile.

    A candidate's profile is the mean representation of its
    associated movies. The target profile is also a mean
    representation, calculated using the same method.
    """
    if not candidate_indices or not target_indices:
        return 0.0

    target = profile_vector(target_indices, method)
    candidate = profile_vector(candidate_indices, method)

    if target is None or candidate is None:
        return 0.0

    return float(cosine_similarity(target, candidate)[0, 0])


def blended_score(similarity, rating_score):
    """Apply identical similarity/rating weights to both methods."""
    return (
        SIMILARITY_WEIGHT * similarity
        + RATING_WEIGHT * rating_score
    )


def mean_rating_score(indices):
    """Mean normalized rating score for a group of movies."""
    if not indices:
        return 0.0

    return float(df.iloc[indices]["rating_score"].mean())


def mean_weighted_rating(indices):
    """Mean original weighted rating for a group of movies."""
    if not indices:
        return np.nan

    return float(df.iloc[indices]["weightedRating"].mean())


def get_category_indices(category):
    """Find movies matching the requested category."""
    category = str(category).strip().lower()

    return df.index[
        df["movie_category"].str.lower() == category
    ].tolist()


def query_similarity(
    query_tokens,
    method="KNN"
):
    """
    Calculate each movie's cosine similarity to a query
    containing selected genres, actors, and directors.

    KNN uses the original feature space.
    SVD projects the query into the learned latent space.
    """
    method = method.upper()

    valid_tokens = [
        token for token in query_tokens
        if token in feature_index
    ]

    scores = np.zeros(len(df), dtype=float)

    if not valid_tokens:
        return scores

    query_columns = [
        feature_index[token]
        for token in valid_tokens
    ]

    # Build a binary query vector in the original feature space.
    query = csr_matrix(
        (
            np.ones(len(query_columns)),
            (
                np.zeros(len(query_columns), dtype=int),
                query_columns
            )
        ),
        shape=(1, X.shape[1])
    )

    query = normalize(query, norm="l2", axis=1)

    if method == "KNN":
        scores = cosine_similarity(
            query,
            X_knn
        ).ravel()

    elif method == "SVD":
        query_latent = svd.transform(query)
        query_latent = normalize(
            query_latent,
            norm="l2",
            axis=1
        )

        scores = cosine_similarity(
            query_latent,
            X_svd
        ).ravel()

    else:
        raise ValueError("method must be 'KNN' or 'SVD'")

    # Negative cosine values are not useful as positive
    # recommendation relevance scores.
    return np.clip(scores, 0.0, 1.0)


# Recommendation Functions

In [ ]:
def recommend_genres(
    category,
    method="KNN",
    n=TOP_N_GENRES
):
    """
    Rank genres by similarity to the selected category's
    movie profile, combined with their associated ratings.
    """
    category_indices = get_category_indices(category)

    if not category_indices:
        raise ValueError(
            f"No movies found for category '{category}'. "
            f"Available categories: "
            f"{sorted(df['movie_category'].dropna().unique())}"
        )

    results = []

    for genre, movie_indices in genre_to_indices.items():
        similarity = similarity_to_profile(
            movie_indices,
            category_indices,
            method
        )

        rating = mean_rating_score(movie_indices)

        score = blended_score(similarity, rating)

        results.append({
            "genre": genre,
            "movie_count": len(movie_indices),
            "mean_weighted_rating": mean_weighted_rating(
                movie_indices
            ),
            "similarity": similarity,
            "rating_score": rating,
            "score": score,
        })

    result_df = pd.DataFrame(results)

    return (
        result_df
        .sort_values(
            ["score", "movie_count"],
            ascending=[False, False]
        )
        .head(n)
        .reset_index(drop=True)
    )


def recommend_people(
    genre,
    category,
    person_type="actor",
    method="KNN",
    n=TOP_N_PEOPLE
):
    """
    Rank actors or directors based on:
      1. Similarity to the selected category + genre profile.
      2. Mean rating score of their associated movies.

    If the selected category has no movies in the genre,
    fall back to all movies belonging to that genre.
    """
    genre = str(genre).strip().lower()
    person_type = person_type.strip().lower()

    if person_type not in {"actor", "director"}:
        raise ValueError(
            "person_type must be 'actor' or 'director'"
        )

    genre_indices = genre_to_indices.get(genre, [])

    if not genre_indices:
        return pd.DataFrame()

    category_indices = set(get_category_indices(category))

    # Prefer movies that match BOTH category and genre.
    target_indices = [
        i for i in genre_indices
        if i in category_indices
    ]

    # Fallback if the category/genre intersection is empty.
    if not target_indices:
        target_indices = genre_indices

    if person_type == "actor":
        person_map = actor_to_indices
    else:
        person_map = director_to_indices

    results = []

    for person, person_indices in person_map.items():
        # Only consider people who have at least one movie
        # associated with the genre being recommended.
        relevant_indices = [
            i for i in person_indices
            if i in set(genre_indices)
        ]

        if not relevant_indices:
            continue

        similarity = similarity_to_profile(
            relevant_indices,
            target_indices,
            method
        )

        rating = mean_rating_score(relevant_indices)

        score = blended_score(similarity, rating)

        results.append({
            "person": person,
            "person_type": person_type,
            "genre": genre,
            "movie_count": len(relevant_indices),
            "mean_weighted_rating": mean_weighted_rating(
                relevant_indices
            ),
            "similarity": similarity,
            "rating_score": rating,
            "score": score,
        })

    if not results:
        return pd.DataFrame()

    return (
        pd.DataFrame(results)
        .sort_values(
            ["score", "movie_count"],
            ascending=[False, False]
        )
        .head(n)
        .reset_index(drop=True)
    )

def recommend_movies(
    genres,
    actors,
    directors,
    method="KNN",
    n=TOP_N_MOVIES
):
    """
    Rank movies by similarity to selected genres, actors,
    and directors, combined with normalized weighted ratings.

    Movies may belong to any category.
    """
    genre_tokens = [
        f"genre:{str(g).strip().lower()}"
        for g in genres
    ]

    actor_tokens = [
        f"actor:{str(a).strip().lower()}"
        for a in actors
    ]

    director_tokens = [
        f"director:{str(d).strip().lower()}"
        for d in directors
    ]

    groups = [
        (genre_tokens, GENRE_WEIGHT),
        (actor_tokens, ACTOR_WEIGHT),
        (director_tokens, DIRECTOR_WEIGHT),
    ]

    # Calculate each feature group's similarities separately.
    group_scores = []
    active_weights = []

    for tokens, weight in groups:
        if any(token in feature_index for token in tokens):
            scores = query_similarity(tokens, method)
            group_scores.append(scores * weight)
            active_weights.append(weight)

    if not active_weights:
        raise ValueError(
            "None of the selected genres, actors, or directors "
            "were found in the fitted feature vocabulary."
        )

    total_weight = sum(active_weights)

    combined_similarity = (
        np.sum(group_scores, axis=0) / total_weight
    )

    rating_scores = df["rating_score"].to_numpy()

    final_scores = (
        SIMILARITY_WEIGHT * combined_similarity
        + RATING_WEIGHT * rating_scores
    )

    # Require at least one exact selected-feature match.
    # This prevents unrelated movies from entering solely
    # because their rating is high.
    matched_indices = set()

    for token in (
        genre_tokens + actor_tokens + director_tokens
    ):
        if token not in feature_index:
            continue

        column = feature_index[token]
        matching_movies = X[:, column].nonzero()[0]
        matched_indices.update(matching_movies.tolist())

    if not matched_indices:
        return pd.DataFrame()

    candidate_indices = np.array(
        sorted(matched_indices),
        dtype=int
    )

    result = df.iloc[candidate_indices][
        [
            "tconst",
            "primaryTitle",
            "movie_category",
            "genres",
            "weightedRating",
        ]
    ].copy()

    result["feature_similarity"] = combined_similarity[
        candidate_indices
    ]

    result["final_score"] = final_scores[candidate_indices]

    result = result.sort_values(
        ["final_score", "feature_similarity", "weightedRating"],
        ascending=[False, False, False]
    )

    return result.head(n).reset_index(drop=True)

# Overall Recommendation Pipeline

In [ ]:
def run_recommendation_pipeline(
    category,
    method="KNN",
    n_genres=TOP_N_GENRES,
    n_people=TOP_N_PEOPLE,
    n_movies=TOP_N_MOVIES,
    verbose=True
):
    """
    Run genre, actor, director, and movie recommendations
    using either KNN or SVD.
    """
    method = method.upper()

    if method not in {"KNN", "SVD"}:
        raise ValueError("method must be 'KNN' or 'SVD'")

    # Stage 1: genres.
    genres_df = recommend_genres(
        category=category,
        method=method,
        n=n_genres
    )

    # Stage 2: actors and directors for each genre.
    people_results = {}

    for genre in genres_df["genre"]:
        people_results[genre] = {
            "actors": recommend_people(
                genre=genre,
                category=category,
                person_type="actor",
                method=method,
                n=n_people
            ),
            "directors": recommend_people(
                genre=genre,
                category=category,
                person_type="director",
                method=method,
                n=n_people
            ),
        }

    # Use all recommended entities as the movie query.
    selected_genres = genres_df["genre"].tolist()

    selected_actors = []
    selected_directors = []

    for genre, people in people_results.items():
        selected_actors.extend(
            people["actors"]["person"].tolist()
            if not people["actors"].empty else []
        )

        selected_directors.extend(
            people["directors"]["person"].tolist()
            if not people["directors"].empty else []
        )

    # Remove duplicates while preserving order.
    selected_actors = list(dict.fromkeys(selected_actors))
    selected_directors = list(dict.fromkeys(selected_directors))

    # Stage 3: movies.
    movies_df = recommend_movies(
        genres=selected_genres,
        actors=selected_actors,
        directors=selected_directors,
        method=method,
        n=n_movies
    )

    output = {
        "method": method,
        "category": category,
        "genres": genres_df,
        "people": people_results,
        "movies": movies_df,
    }

    if verbose:
        print(f"\n{'=' * 65}")
        print(f"{method} RECOMMENDATIONS | CATEGORY: {category}")
        print(f"{'=' * 65}")

        print("\nTOP GENRES")
        print(
            genres_df[
                [
                    "genre",
                    "movie_count",
                    "mean_weighted_rating",
                    "similarity",
                    "score",
                ]
            ].round(3).to_string(index=False)
        )

        for genre, people in people_results.items():
            print(f"\nGENRE: {genre.upper()}")

            print("\n  Actors")
            if people["actors"].empty:
                print("  No actors found.")
            else:
                print(
                    people["actors"][
                        [
                            "person",
                            "movie_count",
                            "mean_weighted_rating",
                            "similarity",
                            "score",
                        ]
                    ].round(3).to_string(index=False)
                )

            print("\n  Directors")
            if people["directors"].empty:
                print("  No directors found.")
            else:
                print(
                    people["directors"][
                        [
                            "person",
                            "movie_count",
                            "mean_weighted_rating",
                            "similarity",
                            "score",
                        ]
                    ].round(3).to_string(index=False)
                )

        print("\nTOP MOVIES")
        if movies_df.empty:
            print("No matching movies found.")
        else:
            print(
                movies_df[
                    [
                        "primaryTitle",
                        "movie_category",
                        "genres",
                        "weightedRating",
                        "feature_similarity",
                        "final_score",
                    ]
                ].round(3).to_string(index=False)
            )

    return output

In [ ]:
category = "AAA"

knn_results = run_recommendation_pipeline(
    category=category,
    method="KNN"
)

svd_results = run_recommendation_pipeline(
    category=category,
    method="SVD"
)

In [ ]:
genre_comparison = (
    knn_results["genres"][
        ["genre", "similarity", "score"]
    ]
    .merge(
        svd_results["genres"][
            ["genre", "similarity", "score"]
        ],
        on="genre",
        how="outer",
        suffixes=("_KNN", "_SVD")
    )
    .sort_values("score_KNN", ascending=False)
)

print(genre_comparison.round(3))

In [ ]:
movie_comparison = pd.merge(
    knn_results["movies"][
        [
            "primaryTitle",
            "feature_similarity",
            "final_score",
        ]
    ],
    svd_results["movies"][
        [
            "primaryTitle",
            "feature_similarity",
            "final_score",
        ]
    ],
    on="primaryTitle",
    how="outer",
    suffixes=("_KNN", "_SVD")
)

print(movie_comparison.round(3))

# 4. Evaluations

## 4.1 Splitting the data

In [ ]:
data = df.copy()

data = data[
    data.apply(
        lambda r: bool(
            r["genre_set"]
            or r["actor_set"]
            or r["director_set"]
        ),
        axis=1
    )
].reset_index(drop=True)

if len(data) < 10:
    raise ValueError("Too few movies for a meaningful hold-out evaluation.")


all_indices = np.arange(len(data))

train_indices, test_indices = train_test_split(
    all_indices,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    shuffle=True,
)

train = data.iloc[train_indices].reset_index(drop=True)
test = data.iloc[test_indices].reset_index(drop=True)

print(f"Training movies: {len(train):,}")
print(f"Test movies:     {len(test):,}")

## 4.2 Build Training Features

In [ ]:
def make_tokens(row):
    tokens = (
        [f"genre:{g}" for g in row["genre_set"]]
        + [f"actor:{a}" for a in row["actor_set"]]
        + [f"director:{d}" for d in row["director_set"]]
    )
    return tokens

train["tokens"] = train.apply(make_tokens, axis=1)

encoder = MultiLabelBinarizer(sparse_output=True)
X_train = encoder.fit_transform(train["tokens"]).tocsr()

feature_index = {
    token: i for i, token in enumerate(encoder.classes_)
}

X_train_knn = normalize(X_train, norm="l2", axis=1)

n_components = min(
    N_COMPONENTS,
    X_train.shape[0] - 1,
    X_train.shape[1] - 1,
)

if n_components < 1:
    raise ValueError("Not enough training rows or features for SVD.")

svd = TruncatedSVD(
    n_components=n_components,
    random_state=RANDOM_STATE,
)

X_train_svd = normalize(
    svd.fit_transform(X_train),
    norm="l2",
    axis=1,
)


# Normalize ratings using TRAINING data only.
rating_median = train["weightedRating"].median()
if pd.isna(rating_median):
    rating_median = 0.0

train_ratings = train["weightedRating"].fillna(rating_median)

rating_min = train_ratings.min()
rating_max = train_ratings.max()

if rating_max > rating_min:
    train["rating_score"] = (
        (train_ratings - rating_min) / (rating_max - rating_min)
    )
else:
    train["rating_score"] = 0.5

## 4.3 Similarity and Recomm Functions

In [ ]:

def query_tokens(row):
    """Features of a held-out query movie."""
    return (
        [f"genre:{g}" for g in row["genre_set"]]
        + [f"actor:{a}" for a in row["actor_set"]]
        + [f"director:{d}" for d in row["director_set"]]
    )


def group_tokens(row, group):
    if group == "genre":
        values = row["genre_set"]
    elif group == "actor":
        values = row["actor_set"]
    elif group == "director":
        values = row["director_set"]
    else:
        raise ValueError(group)

    return [f"{group}:{x}" for x in values]


def query_vector(tokens):
    """Create a query vector using the training vocabulary."""
    valid = [t for t in tokens if t in feature_index]

    if not valid:
        return None

    cols = [feature_index[t] for t in valid]

    vector = csr_matrix(
        (
            np.ones(len(cols)),
            (np.zeros(len(cols), dtype=int), cols)
        ),
        shape=(1, X_train.shape[1]),
    )

    return normalize(vector, norm="l2", axis=1)


def similarity_scores(tokens, method):
    """
    Similarity between one held-out query and every
    training movie.
    """
    q = query_vector(tokens)

    if q is None:
        return np.zeros(len(train), dtype=float)

    if method == "KNN":
        return cosine_similarity(q, X_train_knn).ravel()

    if method == "SVD":
        q_latent = normalize(
            svd.transform(q),
            norm="l2",
            axis=1,
        )
        return cosine_similarity(q_latent, X_train_svd).ravel()

    raise ValueError("method must be 'KNN' or 'SVD'")


def recommend_for_query(query_row, method, k=K):
    """
    Return top-k training movies for a held-out query movie.
    No test movie is included in the candidate pool.
    """
    group_specs = [
        ("genre", GENRE_WEIGHT),
        ("actor", ACTOR_WEIGHT),
        ("director", DIRECTOR_WEIGHT),
    ]

    group_results = []
    active_weights = []

    for group, weight in group_specs:
        tokens = group_tokens(query_row, group)

        # Only activate a group if at least one feature
        # exists in the training vocabulary.
        if any(t in feature_index for t in tokens):
            group_results.append(
                weight * similarity_scores(tokens, method)
            )
            active_weights.append(weight)

    if not active_weights:
        return pd.DataFrame()

    similarity = np.sum(group_results, axis=0) / sum(active_weights)

    final_score = (
        SIMILARITY_WEIGHT * similarity
        + RATING_WEIGHT * train["rating_score"].to_numpy()
    )

    # Exclude training movies with no overlap with any known
    # query feature, so rating alone cannot create a match.
    tokens = query_tokens(query_row)
    known_tokens = [t for t in tokens if t in feature_index]

    if not known_tokens:
        return pd.DataFrame()

    candidate_mask = np.zeros(len(train), dtype=bool)

    for token in known_tokens:
        col = feature_index[token]
        candidate_mask |= np.asarray(
            X_train[:, col].toarray()
        ).ravel().astype(bool)

    candidate_idx = np.flatnonzero(candidate_mask)

    if len(candidate_idx) == 0:
        return pd.DataFrame()

    recs = train.iloc[candidate_idx][
        [
            "tconst",
            "primaryTitle",
            "genres",
            "movie_category",
            "weightedRating",
        ]
    ].copy()

    recs["similarity"] = similarity[candidate_idx]
    recs["final_score"] = final_score[candidate_idx]

    return (
        recs.sort_values(
            ["final_score", "similarity"],
            ascending=False,
        )
        .head(k)
        .reset_index(drop=True)
    )


def jaccard_similarity(set_a, set_b):
    union = set_a | set_b

    if not union:
        return 0.0

    return len(set_a & set_b) / len(union)


def relevance_score(query_row, candidate_row):
    """
    Graded relevance from genre, actor, and director overlap.

    Range: 0–1.
    The weights match the feature-group weights.
    """
    return (
        GENRE_WEIGHT * jaccard_similarity(
            query_row["genre_set"], candidate_row["genre_set"]
        )
        + ACTOR_WEIGHT * jaccard_similarity(
            query_row["actor_set"], candidate_row["actor_set"]
        )
        + DIRECTOR_WEIGHT * jaccard_similarity(
            query_row["director_set"], candidate_row["director_set"]
        )
    )


def is_relevant(query_row, candidate_row, threshold=0.10):
    return relevance_score(query_row, candidate_row) >= threshold


def jaccard(a, b):
    union = a | b
    return len(a & b) / len(union) if union else 0.0


def relevance(query, candidate):
    return sum(
        weight * jaccard(query[col], candidate[col])
        for weight, col in GROUPS.values()
    )


train_records = train.to_dict("records")


def query_tokens_for_group(query, group):
    col = GROUPS[group][1]
    return [f"{group}:{x}" for x in query[col]]


def group_similarity(query, group, method):
    tokens = query_tokens_for_group(query, group)
    valid = [t for t in tokens if t in feature_index]

    if not valid:
        return None

    cols = [feature_index[t] for t in valid]
    q = csr_matrix(
        (
            np.ones(len(cols)),
            (np.zeros(len(cols), dtype=int), cols)
        ),
        shape=(1, X_train.shape[1])
    )
    q = normalize(q, norm="l2", axis=1)

    if method == "KNN":
        return cosine_similarity(q, X_train_knn).ravel()

    q_latent = normalize(svd.transform(q), norm="l2", axis=1)
    return cosine_similarity(q_latent, X_train_svd).ravel()


def get_similarity_scores(query, method):
    weighted_scores = []
    active_weights = []

    for group, (weight, _) in GROUPS.items():
        scores = group_similarity(query, group, method)

        if scores is not None:
            weighted_scores.append(weight * scores)
            active_weights.append(weight)

    if not active_weights:
        return None

    return np.sum(weighted_scores, axis=0) / sum(active_weights)



def dcg_at_k(relevances, k=10):
    rel = np.asarray(relevances[:k], dtype=float)

    if len(rel) == 0:
        return 0.0

    gains = (2 ** rel) - 1
    discounts = np.log2(np.arange(2, len(rel) + 2))

    return float(np.sum(gains / discounts))


def ndcg_at_k(ranked_relevances, all_relevances, k=10):
    ideal = sorted(all_relevances, reverse=True)[:k]
    ideal_dcg = dcg_at_k(ideal, k)

    if ideal_dcg == 0:
        return np.nan

    return dcg_at_k(ranked_relevances, k) / ideal_dcg

## 4.4 Evaluation Method

In [ ]:

def evaluate_method(method, k=10):
    ndcg_scores = []
    feature_similarities = []
    avg_weighted_ratings = []

    for _, query_series in test.iterrows():
        query = query_series.to_dict()

        similarity = get_similarity_scores(query, method)

        if similarity is None:
            continue

        # Same final ranking formula for both methods.
        final_scores = (
            SIMILARITY_WEIGHT * similarity
            + RATING_WEIGHT * train["rating_score"].to_numpy()
        )

        # Only recommend movies sharing at least one known feature.
        tokens = (
            query_tokens_for_group(query, "genre")
            + query_tokens_for_group(query, "actor")
            + query_tokens_for_group(query, "director")
        )
        known_tokens = [t for t in tokens if t in feature_index]

        if not known_tokens:
            continue

        candidate_mask = np.zeros(len(train), dtype=bool)

        for token in known_tokens:
            col = feature_index[token]
            candidate_mask |= np.asarray(
                X_train[:, col].toarray()
            ).ravel().astype(bool)

        candidate_indices = np.flatnonzero(candidate_mask)

        if len(candidate_indices) == 0:
            continue

        # Rank candidates by the combined final score.
        ranked_indices = candidate_indices[
            np.argsort(final_scores[candidate_indices])[::-1][:k]
        ]

        # ---------- Metric 1: Mean Feature Similarity@10 ----------
        # Similarity is the combined genre/actor/director similarity.
        feature_similarities.append(
            float(np.mean(similarity[ranked_indices]))
        )

        # ---------- Metric 2: Mean Weighted Rating@10 ----------
        # Use original ratings, not normalized rating_score.
        top_ratings = train.iloc[ranked_indices]["weightedRating"]
        avg_weighted_ratings.append(
            float(top_ratings.mean())
            if top_ratings.notna().any()
            else np.nan
        )

        # ---------- Metric 3: NDCG@10 ----------
        ranked_relevance = [
            relevance(query, train_records[i])
            for i in ranked_indices
        ]

        all_relevance = [
            relevance(query, train_records[i])
            for i in candidate_indices
        ]

        if max(all_relevance, default=0.0) == 0:
            continue

        score = ndcg_at_k(
            ranked_relevance,
            all_relevance,
            k
        )

        if not np.isnan(score):
            ndcg_scores.append(score)

    return {
        "method": method,
        "NDCG@10": (
            float(np.mean(ndcg_scores))
            if ndcg_scores else np.nan
        ),
        "Mean Feature Similarity@10": (
            float(np.mean(feature_similarities))
            if feature_similarities else np.nan
        ),
        "Mean Average Weighted Rating@10": (
            float(np.mean(avg_weighted_ratings))
            if avg_weighted_ratings else np.nan
        ),
        "Evaluated Queries": len(ndcg_scores),
    }



In [ ]:
results = pd.DataFrame([
    evaluate_method("KNN", k=10),
    evaluate_method("SVD", k=10),
])

print(results.round(4).to_string(index=False))


# 5. Model Saving

In [ ]:
def save_model(path=MODEL_PATH):
    """Save all fitted objects required for recommendations."""

    artifacts = {
        "df": df,
        "encoder": encoder,
        "feature_index": feature_index,
        "X": X,
        "X_knn": X_knn,
        "svd": svd,
        "X_svd": X_svd,

        "genre_to_indices": genre_to_indices,
        "actor_to_indices": actor_to_indices,
        "director_to_indices": director_to_indices,

        "SIMILARITY_WEIGHT": SIMILARITY_WEIGHT,
        "RATING_WEIGHT": RATING_WEIGHT,
        "GENRE_WEIGHT": GENRE_WEIGHT,
        "ACTOR_WEIGHT": ACTOR_WEIGHT,
        "DIRECTOR_WEIGHT": DIRECTOR_WEIGHT,
        "TOP_N_GENRES": TOP_N_GENRES,
        "TOP_N_PEOPLE": TOP_N_PEOPLE,
        "TOP_N_MOVIES": TOP_N_MOVIES,
    }

    joblib.dump(artifacts, path, compress=3)
    print(f"Model saved to: {path}")


def load_model(path=MODEL_PATH):
    """
    Load saved artifacts into this module's namespace.

    Call this before run_recommendation_pipeline().
    """
    global df, encoder, feature_index, X, X_knn, svd, X_svd
    global genre_to_indices, actor_to_indices, director_to_indices
    global SIMILARITY_WEIGHT, RATING_WEIGHT
    global GENRE_WEIGHT, ACTOR_WEIGHT, DIRECTOR_WEIGHT
    global TOP_N_GENRES, TOP_N_PEOPLE, TOP_N_MOVIES

    artifacts = joblib.load(path)

    globals().update(artifacts)

    print(f"Model loaded from: {path}")
    return True
